In [5]:
import pandas as pd
import nltk
import re

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Step 1: Load Required Resources
nltk.download('stopwords')

# Step 2: Load Dataset
data = pd.read_csv("Musical_instruments_reviews2.csv")

print(data.head())

# Step 3: Create Sentiment Labels
data["Sentiment"] = data["division"].str.capitalize()

# Remove missing reviews
data = data.dropna(subset=["reviewText"])

# Step 4: Text Preprocessing
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.split()

    words = [stemmer.stem(word)
             for word in text
             if word not in stop_words]

    return " ".join(words)

# Clean the review text
data["Clean_Text"] = data["reviewText"].apply(preprocess)

print(data[["reviewText", "Clean_Text", "Sentiment"]].head())

# Step 5: Feature Extraction using TF-IDF
tfidf = TfidfVectorizer()

X = tfidf.fit_transform(data["Clean_Text"])

y = data["Sentiment"]

# Step 6: Split Dataset into Training and Testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Step 7: Train Sentiment Model using Naive Bayes
model = MultinomialNB()

model.fit(
    X_train,
    y_train
)

# Step 8: Test Model
prediction = model.predict(X_test)

print("\nPredictions:")
print(prediction)

print("\nAccuracy:")
print(accuracy_score(y_test, prediction))

print("\nClassification Report:")
print(classification_report(y_test, prediction, zero_division=0))

# Step 9: Predict New Review
new_review = [
    "This product is fantastic and I really love it"
]

new_review = [preprocess(review) for review in new_review]

new_text = tfidf.transform(new_review)

result = model.predict(new_text)

print("\nPrediction:")
print(result)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


       reviewerID        asin  \
0  A2IBPI20UZIR0U  1384719342   
1  A14VAT5EAX3D9S  1384719342   
2  A195EZSQDW3E21  1384719342   
3  A2C00NNG1ZQQG2  1384719342   
4   A94QU4C90B1AX  1384719342   

                                       reviewerName   helpful  \
0  cassandra tu "Yeah, well, that's just like, u...    [0, 0]   
1                                              Jake  [13, 14]   
2                     Rick Bennette "Rick Bennette"    [1, 1]   
3                         RustyBill "Sunday Rocker"    [0, 0]   
4                                     SEAN MASLANKA    [0, 0]   

                                          reviewText  overall  \
0  Not much to write about here, but it does exac...        5   
1  The product does exactly as it should and is q...        5   
2  The primary job of this device is to block the...        5   
3  Nice windscreen protects my MXL mic and preven...        5   
4  This pop filter is great. It looks and perform...        5   

                   

In [2]:
print(data.columns)

Index(['reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText',
       'overall', 'summary', 'unixReviewTime', 'reviewTime', 'division'],
      dtype='object')


In [4]:
print(data["Sentiment"].value_counts())

Sentiment
Positive    5898
Neutral      772
Negative     467
Name: count, dtype: int64
